# rhino-poc — Colab

**Baslamadan:** Runtime > Change runtime type > **GPU (T4)**.

Hucreleri **sirayla** kos. Atlama — 2. hucre degiskenleri tanimlar,
sonraki hucreler onlari kullanir (`NameError` alirsan 2. hucreyi kosmamissindir).

Colab burada sadece **GPU gereken adim** icin: COLMAP MVS. Bkz. `PLAN.md` SS4.

In [ ]:
!nvidia-smi

## 1. Kurulum — repo + Drive + yollar
Tek hucre, her seyi tanimlar. Oturum dusarse **bu hucreden** devam et.

In [ ]:
import os

REPO = '/content/rhino-poc'
WORK = '/content/work'          # calisma diski (HIZLI)

if os.path.isdir(REPO + '/.git'):
    !cd {REPO} && git fetch --quiet origin && git reset --hard origin/main
else:
    !rm -rf {REPO}
    !git clone --quiet https://github.com/Daml4Yilmaz/rhino-poc.git {REPO}

from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/rhino-poc-data'   # GIRDI: video/yakalama
SAVE = '/content/drive/MyDrive/rhino-poc-out'    # CIKTI: mesh buraya kaydedilir
os.makedirs(DATA, exist_ok=True)
os.makedirs(SAVE, exist_ok=True)
os.makedirs(WORK, exist_ok=True)

!cd {REPO} && git log --oneline -1
print('DATA icerigi:', os.listdir(DATA))

**Neden `/content/work`?** COLMAP MVS on binlerce kucuk dosya yazar.
Drive FUSE uzerinden bu cok yavastir ve kopabilir. Hesaplama hizli diskte kosar,
**sonuclar** son hucrede Drive'a kopyalanir.

## 2. COLMAP (CUDA'li)
`%%bash` hucresi Python degiskenlerini GOREMEZ — burada yol yazmiyoruz, sorun degil.

In [ ]:
%%bash
cd /opt
if [ ! -x /opt/bin/micromamba ]; then
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
fi
if [ ! -x /opt/colmapenv/bin/colmap ]; then
  /opt/bin/micromamba create -y -q -p /opt/colmapenv -c conda-forge 'colmap=*=gpu*'
fi

In [ ]:
import os
for c in ('/opt/colmapenv/bin/colmap', '/usr/local/colmap/bin/colmap'):
    if os.path.exists(c):
        os.system('ln -sf ' + c + ' /usr/local/bin/colmap')
        break
!colmap -h 2>&1 | head -3

Ciktida **`with CUDA`** yazmali. `without CUDA` yazarsa MVS adimi kosmaz.

**PATH uyarisi:** conda klasorunu PATH'in basina EKLEME — icindeki python sistem
python'unu golgeler ve `cv2` kirilir. Sadece symlink (ustteki hucre bunu yapiyor).

## 3. Python paketleri

In [ ]:
!pip install -q opencv-contrib-python open3d trimesh pycolmap typer pandas pillow
!pip install -q {REPO}
!python -m poc.cli --help | head -12

`poc` yerine **`python -m poc.cli`** kullaniyoruz: kurulan komut bazen Colab'in
PATH'ine girmez, modul cagrisi her zaman calisir.

**`-e` (editable) KULLANMA** — Jupyter ayni oturumda import edemez.
Repoda kod degisirse 1. ve bu hucreyi tekrar kos.

## 4A. TEST — duz .mov / .mp4

Videoyu Drive'a koy: `MyDrive/rhino-poc-data/test.mov`

Sadece **fotogrametri hattini** dogrular: kareler -> SfM -> MVS -> mesh.
**Model BIRIMSIZ olur** (`model_unitless.glb`) — aci ve Goode orani anlamli,
mm cinsinden uzunluk/genislik/sapma URETILEMEZ. Onun icin 4B gerekir.

Ilk denemede `--n-frames 150` birak: MVS kare sayisiyla dogru orantili,
150 kare T4'te ~15-25 dk, 300 kare bir saati asabilir.

In [ ]:
VIDEO = DATA + '/test.mov'
OUT   = WORK + '/test_out'
!python -m poc.cli process {VIDEO} --out {OUT} --n-frames 150 --max-dim 1600

## 4B. GERCEK — Stray Scanner yakalamasi

Klasoru oldugu gibi Drive'a kopyala: `MyDrive/rhino-poc-data/vaka_001_stray/`
icinde `rgb.mp4`, `odometry.csv`, `camera_matrix.csv`, `depth/`, `confidence/`.

Bu yolda olcek de kosar. Beklenen: `agreement_pct` < 1.5, `scale_verified: true`.

In [ ]:
CAPTURE = DATA + '/vaka_001_stray'
OUT     = WORK + '/vaka_001'
!python -m poc.cli process {CAPTURE} --out {OUT} --n-frames 300

## 5. Sonuclari Drive'a kaydet
Yukarida hangi hucreyi kostuysan (`OUT` ondan gelir) bunu kos.

In [ ]:
import os, shutil, json

case = os.path.basename(OUT)
dst  = os.path.join(SAVE, case)
os.makedirs(dst, exist_ok=True)

KEEP = ['mesh_raw.ply', 'model.glb', 'model_unitless.glb',
        'scale.json', 'measurements.json', 'frames_index.json',
        'landmarks.json']
for f in KEEP:
    p = os.path.join(OUT, f)
    if os.path.exists(p):
        shutil.copy2(p, dst)
        print('kaydedildi: %-22s %8.1f MB' % (f, os.path.getsize(p)/1e6))

print('\nDrive konumu:', dst)
sj = os.path.join(dst, 'scale.json')
if os.path.exists(sj):
    print(json.dumps(json.load(open(sj)), indent=2))

`mesh_raw.ply` ham yuzey, `model.glb` / `model_unitless.glb` goruntuleyici icin.
GLB'yi [gltf.report](https://gltf.report) veya Blender'da ac.
4B'de kafa bbox ~200-250 mm cikmali.

COLMAP ara ciktilari (`colmap/`, `frames/`) bilerek kopyalanmaz — GB'larca yer tutar.
Gerekirse: `!cp -r {OUT}/colmap {dst}/`